# ST-01 — Siretisation Phase 1 (SIRET exact)

Pour chaque EG FINESS ayant un `nmsiret_stru`, lookup direct dans la base Etab SIRENE complète. Calcul du score (nom + adresse) et classification.

Statuts produits : VALIDE_FORT / VALIDE / DOUTEUX / REJETE / SANS_SIRET / SIRET_INCONNU

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.siretisation import matching_direct_siret
from src.excel_export import export_phase1_excel, LABELS
from src.display      import afficher_tableau, afficher_synthese
from config.settings  import (
    FINESS_EG_CLEAN, SIRENE_ETAB_CLEAN, ST_PHASE1, RESULTS_ST_DIR,
)

RESULTS_ST_DIR.mkdir(parents=True, exist_ok=True)

## 1. Chargement

In [2]:
df_eg   = pd.read_parquet(FINESS_EG_CLEAN)
df_etab = pd.read_parquet(SIRENE_ETAB_CLEAN)

df_eg['nmsiret_stru'] = df_eg['nmsiret_stru'].fillna('').astype(str)
df_etab['siret']      = df_etab['siret'].astype(str)

print(f'EG FINESS  : {len(df_eg):,}')
print(f'Etab SIRENE: {len(df_etab):,}')

IOStream.flush timed out


EG FINESS  : 104,805
Etab SIRENE: 16,796,675


## 2. Matching SIRET exact

In [3]:
df_resultats = matching_direct_siret(df_eg, df_etab, desc='Matching SIRET exact')
print(df_resultats['statut'].value_counts())

Matching SIRET exact:   0%|          | 0/104805 [00:00<?, ?it/s]

statut
VALIDE           34402
VALIDE_FORT      25856
SANS_SIRET       13124
SIRET_INCONNU    12357
DOUTEUX          10804
REJETE            8262
Name: count, dtype: int64


## 3. Enrichissement avec colonnes Etab SIRENE

In [4]:
COLS_ETAB_JOIN = ['siret', 'denominationUniteLegale', 'sigleUniteLegale',
                  'enseigne1Etablissement', 'enseigne2Etablissement',
                  'enseigne3Etablissement', 'denominationUsuelleEtablissement',
                  'adresse_complete_etab', 'codeCommuneEtablissement',
                  'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale']
etab_join = df_etab[[c for c in COLS_ETAB_JOIN if c in df_etab.columns]].drop_duplicates('siret').copy()
etab_join['siret'] = etab_join['siret'].astype(str)

df_resultats['siret_etab'] = df_resultats['siret_etab'].astype(str)
df_resultats = df_resultats.merge(
    etab_join, left_on='siret_etab', right_on='siret', how='left',
).drop(columns=['siret'], errors='ignore')

## 4. Aperçu

In [5]:
afficher_tableau(
    df_resultats[df_resultats['statut'].isin(['VALIDE_FORT', 'VALIDE'])],
    'Aperçu validés', max_lignes=9,
    colonnes=['idstructure_stru', 'raisonsociale_stru',
              'denominationUniteLegale', 'nom_etab_retenu',
              'score_nom', 'score_adresse', 'score_global', 'statut'],
)

idstructure_stru,raisonsociale_stru,denominationUniteLegale,nom_etab_retenu,score_nom,score_adresse,score_global,statut
2418252,INST. SUP. RÉÉDUCATION PSYCHOMOTRICE,ISRP VICHY,ISRP VICHY,24.790000,100.000000,69.920000,VALIDE
2418253,MAISON DE SANTE DES TROIS RIVIERES,MSP LES 3 RIVIERES,MSP 3 RIVIERES,47.930000,100.000000,79.170000,VALIDE
2418255,MAISON DE SANTE HTES VALLEES D'ARDECHE,SISA DES HAUTES VALLEES D'ARDECHE,SISA HAUTES VALLEES ARDECHE,71.210000,100.000000,88.490000,VALIDE_FORT
2418256,MSP DE BUZANCY,SISA BUZANCY,SISA BUZANCY,56.170000,100.000000,82.470000,VALIDE
2418257,SAAD AAD 09,AAD 09,AAD 09,73.410000,100.000000,89.360000,VALIDE_FORT
2418258,DOMALIANCE FOIX,APM ARIEGE PYRENEES MULTISERVICES,DOMALIANCE FOIX,100.000000,100.000000,100.000000,VALIDE_FORT
2418260,SAAD ASSOCIATION BLEU PRINTEMPS,ASSOCIATION BLEU PRINTEMPS,BLEU PRINTEMPS,78.550000,100.000000,91.420000,VALIDE_FORT
2418261,SAAD A DEUX MAINS,A DEUX MAINS,DEUX MAINS,76.800000,100.000000,90.720000,VALIDE_FORT
2418265,SAAD AAA,A.A.A. AIDES ET ACCOMPAGNEMENT A L'AUTONOMIE,AIDES ACCOMPAGNEMENT AUTONOMIE,27.370000,100.000000,70.950000,VALIDE


## 5. Export Excel

In [6]:
COLS_COMPLET = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru', 'categetab_stru',
    'nmsiret_stru', 'raisonsociale_stru',
    'cdcommune_stru', 'adresse_complete_eg',
    'siret_etab', 'denominationUniteLegale', 'sigleUniteLegale',
    'enseigne1Etablissement', 'enseigne2Etablissement', 'enseigne3Etablissement',
    'denominationUsuelleEtablissement', 'nom_etab_retenu',
    'adresse_complete_etab', 'codeCommuneEtablissement',
    'categorieJuridiqueUniteLegale', 'activitePrincipaleUniteLegale',
    'score_nom', 'score_adresse', 'score_global',
]
COLS_INFO = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru', 'categetab_stru',
    'nmsiret_stru', 'raisonsociale_stru',
    'cdcommune_stru', 'adresse_complete_eg',
]

compteurs = export_phase1_excel(
    df_resultats, ST_PHASE1, COLS_COMPLET, COLS_INFO,
    statuts_score=['VALIDE_FORT', 'VALIDE', 'DOUTEUX', 'REJETE'],
    statuts_info =['SANS_SIRET', 'SIRET_INCONNU'],
)

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  'Synthèse Siretisation Phase 1')
print(f'\nFichier : {ST_PHASE1}')

Statut,Nb,% du total
Valide_fort,"25,856",24.7%
Valide,"34,402",32.8%
Douteux,"10,804",10.3%
Rejeté,"8,262",7.9%
Sans_SIRET,"13,124",12.5%
SIRET_inconnu,"12,357",11.8%
TOTAL,"104,805",100.0%



Fichier : /home/jovyan/work/projet_finess_sirene/results/siretisation/siretisation_phase1.xlsx
